In [1]:
!pip install deep-sort-realtime

  Obtaining dependency information for deep-sort-realtime from https://files.pythonhosted.org/packages/eb/3c/73c13af1dddc287cb42493ad59bd296daef106408e0006a4897c20a4ddd6/deep_sort_realtime-1.3.2-py3-none-any.whl.metadata
  Obtaining dependency information for scipy from https://files.pythonhosted.org/packages/2b/54/9a9edb45345bd6744da5ddfb6628e5d5185920494c6a67ec45b6381004cb/scipy-1.18.0-cp312-cp312-win_amd64.whl.metadata
  Using cached scipy-1.18.0-cp312-cp312-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB 653.6 kB/s eta 0:00:13
   ---------------------------------------- 0.0/8.4 MB 653.6 kB/s eta 0:00:13
   ---------------------------------------- 0.0/8.4 MB 653.6 kB/s eta 0:00:13
   ---------------------------------------- 0.1/8.4 MB 281.8 kB/s eta 0:00:30
   ---------------------------------------- 0.1/8.4 M


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import cv2
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from sklearn.cluster import KMeans
import numpy as np

In [2]:
# Load YOLO model
model = YOLO(r"..\..\detection\fine_tunning_3\runs\detect\train\weights\best.pt")

In [3]:
# Initialize DeepSORT
tracker = DeepSort(
    max_age=50,
    n_init=1,
    max_iou_distance=0.9,
    max_cosine_distance=0.6,
    nn_budget=100,
    bgr=True
)

C:\Users\Mehdi\AppData\Roaming\Python\Python313\site-packages\deep_sort_realtime\embedder\embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Open video
cap = cv2.VideoCapture("../../../data/videos/video1.mp4")

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    r".\tracking_result2.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (frame_width, frame_height)
)

In [5]:
import cv2
import numpy as np

def get_features_for_close_colors(crop, bins=(16, 16)):
    if crop is None:
        print("crop is None")
        return None

    if crop.size == 0:
        print("crop is empty")
        return None

    h, w = crop.shape[:2]
    print(f"Crop size: {w} x {h}")

    if h < 10 or w < 10:
        print("Crop too small")
        return None

    # 1. RESSERREMENT DRASTIQUE : On cible uniquement le HAUT du maillot (Torse haut / Poitrine)
    # On évite ainsi absolument le short, la ceinture et le bas des bras
    y1 = int(h * 0.15)
    y2 = int(h * 0.45)
    x1 = int(w * 0.25)
    x2 = int(w * 0.75)

    high_torso = crop[y1:y2, x1:x2]
    print("High Torso shape:", high_torso.shape)

    if high_torso.size == 0:
        print("High Torso empty")
        return None

    # 2. Élimination de la pelouse en HSV
    hsv_torso = cv2.cvtColor(high_torso, cv2.COLOR_BGR2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    
    green_mask = cv2.inRange(hsv_torso, lower_green, upper_green)
    non_green_mask = cv2.bitwise_not(green_mask)

    # 3. Conversion en LAB pour la stabilité des couleurs
    lab_torso = cv2.cvtColor(high_torso, cv2.COLOR_BGR2LAB)

    # Sécurité si 100% de pelouse détectée
    if cv2.countNonZero(non_green_mask) == 0:
        print("Torso contains only green pelouse!")
        non_green_mask = None

    # 4. SOLUTION AUX COULEURS PROCHES : L'histogramme 2D des canaux A et B
    # Au lieu d'une simple moyenne qui écrase les détails, on calcule la répartition 
    # des couleurs pures (A = Vert/Rouge, B = Bleu/Jaune). On ignore L (la lumière).
    hist = cv2.calcHist(
        [lab_torso],
        [1, 2],              # Canaux A et B uniquement
        non_green_mask,      # On applique le masque anti-pelouse ici
        bins,                # Nombre de paliers (ex: 16x16 = vecteur de 256 valeurs)
        [0, 256, 0, 256]     # Plages de valeurs pour A et B
    )

    # Normalisation pour que la taille de l'image n'influence pas le résultat
    hist = cv2.normalize(hist, hist).flatten()
    print("Robust feature vector shape:", hist.shape)

    return hist

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier # Un excellent choix par défaut
classifier = "null"
player_features = {}      # track_id -> histogram
team_memory = {}          # track_id -> Team 1 / Team 2
clustering_done = False
time =0
while cap.isOpened():
    time = time +1
    ret, frame = cap.read()

    if not ret:
        break

    # ---------------- YOLO ----------------
    results = model(frame, conf=0.3, iou=0.5)

    detections = []

    for result in results:

        for box in result.boxes:

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            conf = float(box.conf[0])
            cls = int(box.cls[0])

            w = x2 - x1
            h = y2 - y1

            detections.append(
                (
                    [x1, y1, w, h],
                    conf,
                    cls
                )
            )

    # ---------------- DeepSORT ----------------
    tracks = tracker.update_tracks(
        detections,
        frame=frame
    )

    # ---------------- Tracking ----------------
    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id

        class_id = track.get_det_class()
        class_name = model.names[class_id]
        print(class_name)

        l, t, r, b = map(int, track.to_ltrb())

        # Empêcher les coordonnées négatives
        l = max(0, l)
        t = max(0, t)
        r = min(frame.shape[1], r)
        b = min(frame.shape[0], b)

        # ---------------- Player Feature Extraction ----------------
        if class_name == "Player":

            if track_id not in player_features:

                clustering_done = False

                player_crop = frame[t:b, l:r]

                if player_crop.size != 0:
                    player_hist = get_features_for_close_colors(player_crop)
                    
                if player_hist is not None:
                    player_features[track_id] = player_hist

        # ---------------- Draw ----------------

        if track_id in team_memory:
            label = f"{team_memory[track_id]} | ID:{track_id}"
        else:
            label = f"{class_name} | ID:{track_id}"

        cv2.rectangle(
            frame,
            (l, t),
            (r, b),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            label,
            (l, t - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

    # ---------------- KMeans ----------------
    # deleting the second condition "and len(player_features) >= 18"
    # ---------------- Classification & Clustering ----------------
    if not clustering_done:
        ids = list(player_features.keys())
        features = np.array(list(player_features.values()))

        if len(features) > 0:
            # Phase d'initialisation : on utilise KMeans
            if time <= 15:
                kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
                labels = kmeans.fit_predict(features)
                
                # Entraînement du Random Forest à la frame 15
                if time == 15:
                    print("Entraînement initial du RandomForestClassifier...")
                    classifier = RandomForestClassifier(n_estimators=100, random_state=42)
                    classifier.fit(features, labels)
            
            # Phase de prédiction (time > 15)
            else:
                # SÉCURITÉ : Si pour une raison quelconque le classifieur est toujours une chaîne de caractères
                if isinstance(classifier, str):
                    print(f"Sécurité activée : Entraînement tardif du RandomForest à la Frame {time}...")
                    kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
                    labels = kmeans.fit_predict(features)
                    
                    classifier = RandomForestClassifier(n_estimators=100, random_state=42)
                    classifier.fit(features, labels)
                else:
                    # Comportement normal si le modèle existe
                    print(f"Prédiction via Random Forest (Frame {time})")
                    labels = classifier.predict(features)

            # Assignation des équipes dans la mémoire globale
            for track_id, cluster in zip(ids, labels):
                if cluster == 0:
                    team_memory[track_id] = "Team A"
                else:
                    team_memory[track_id] = "Team B"

            clustering_done = True
            print("Players clustered successfully!")
    out.write(frame)

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 1 Keeper, 18 Players, 2 Refs, 100.4ms
Speed: 4.1ms preprocess, 100.4ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Ball, 1 Keeper, 18 Players, 2 Refs, 58.5ms
Speed: 6.3ms preprocess, 58.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
Player
Crop size: 20 x 32
High Torso shape: (10, 10, 3)
Torso contains only green pelouse!
Robust feature vector shape: (256,)
Player
Crop size: 21 x 44
High Torso shape: (13, 10, 3)
Robust feature vector shape: (256,)
Player
Crop size: 13 x 34
High Torso shape: (10, 6, 3)
Robust feature vector shape: (256,)
Player
Crop size: 15 x 33
High Torso shape: (10, 8, 3)
Robust feature vector shape: (256,)
Keeper
Player
Crop size: 15 x 43
High Torso shape: (13, 8, 3)
Robust feature vector shape: (256,)
Player
Crop size: 12 x 31
High Torso shape: (9, 6, 3)
Robust feature vector shape: (256,)
Player
Crop size: 13 x 36
High Torso shape: (11, 6, 3)
Robust feature vector shape: (256,)
Player
Crop size: 